# Colab fallback for the take-home exam

Use this notebook only if your laptop cannot run the harness (see TROUBLESHOOTING.md and Section 8 of the handout).
Runtime -> Change runtime type -> **T4 GPU** if available, otherwise CPU (then use `--lite`).

The CLI is identical to the one on your Mac, so the files it writes (`predictions/*.jsonl`, `run_log.txt`, `probe.jsonl`)
are schema-compatible with the validators and with `analysis.py`. Keep the same `--student-id` you use elsewhere.
Colab sessions are wiped when they time out: download the zip from the last cell before closing the tab.


In [ ]:
# 1. Upload student.zip (the handout zip, or a zip of your own student/ directory with your scorers in it)
from google.colab import files
uploaded = files.upload()
zip_name = next(iter(uploaded))
!rm -rf student && mkdir -p student && unzip -q -o "$zip_name" -d student && ls student


In [ ]:
# 2. Install pinned requirements (Colab already ships torch; this only adds what is missing)
%cd /content/student
!pip install -q -r requirements.txt
import torch; print('torch', torch.__version__, 'cuda' if torch.cuda.is_available() else 'cpu')


In [ ]:
# 3. Smoke test (Part 0). Replace S123 with your student id.
STUDENT_ID = 'S123'
%cd /content/student
!python smoke_test.py --student-id $STUDENT_ID
!cat smoke_ok.json


In [ ]:
# 4. One example harness run (Part 2, English, LETTER). Change --lang / --scorer / --part / --variant as needed.
#    Add --lite on a CPU runtime. Every run appends to run_log.txt and LETTER v1_en runs update probe.jsonl.
%cd /content/student
!python -m harness run --part 2 --model qwen2.5 --lang en --scorer LETTER --variant v1_en --student-id $STUDENT_ID --batch-size 8


In [ ]:
# 5. Package the outputs and download them; unzip into your local student/ directory.
%cd /content/student
!zip -q -r colab_outputs.zip predictions run_log.txt probe.jsonl smoke_ok.json -x 'predictions/*.partial'
from google.colab import files
files.download('colab_outputs.zip')


Merging: copy `predictions/*.jsonl` into your local `predictions/`, append the lines of the downloaded `run_log.txt`
(without its header) to your local one, and replace `probe.jsonl` rows only for the (model, lang) pairs you ran here.
Record in the report that these runs were done on Colab (device/dtype are in `run_log.txt`).
